# Week 1 — Python basics

**Goal**: read `data/samples/nagoya_wards.csv`, compute summary stats, and call into `src/spatial_notebooks/stats.py`.

**Done when**: the last cell prints a `Summary` dataclass with row/column/missing counts.

Rule of thumb: if you copy-paste a snippet a second time, move it into `src/` and import it here.

In [1]:
from pathlib import Path  # Path 对象，比字符串拼接更安全、跨平台

from spatial_notebooks.stats import add_density, load_csv, summarize  # 项目封装的三个工具函数

REPO_ROOT = Path.cwd().parent          # notebook 在 notebooks/ 下，上一级即项目根目录
DATA = REPO_ROOT / "data" / "samples" / "nagoya_wards.csv"  # / 运算符拼接路径，等价于 os.path.join
DATA                                   # cell 末尾裸变量名 → Jupyter 自动打印其值

PosixPath('/Users/mac/Projects/Eukarya/spatial-notebooks/data/samples/nagoya_wards.csv')

In [2]:
df = load_csv(DATA)  # 读取 CSV，返回 pandas DataFrame；内部处理了编码与类型推断
df.head()           # 默认显示前 5 行，快速确认数据结构是否符合预期

,ward,population,area_km2,lat,lon
0,Naka,99120,9.38,35.1681,136.9066
1,Higashi,87230,7.71,35.1819,136.9304
2,Kita,165340,17.53,35.2086,136.9161
3,Nishi,153210,17.94,35.1890,136.8771
4,Nakamura,137800,16.30,35.1706,136.8812


In [3]:
summarize(df)  # 返回 Summary dataclass：行数、列数、各列缺失值计数

Summary(rows=16, columns=5, numeric_columns=['population', 'area_km2', 'lat', 'lon'], missing_by_column={'ward': 0, 'population': 0, 'area_km2': 0, 'lat': 0, 'lon': 0})

## Exercises

1. Compute population density (population / area_km2) and rank wards. Hint: `df.assign(density=...)` then `.sort_values`.
2. Which wards have an area larger than the median? Use boolean indexing, not a loop.
3. Add a type hint to a helper you write in the notebook, then extract it into `src/spatial_notebooks/` once the logic stabilizes. Write a pytest test for it.

In [29]:
density_df = (
    df
    .assign(density=df["population"] / df["area_km2"])  # 新增 density 列：人口 ÷ 面积（人/km²）
    .sort_values("density", ascending=False)             # 按密度降序排列，最密集的区排最前
    [["ward", "population", "area_km2", "density"]]      # 只保留 4 列，去掉 lat/lon 让输出更简洁
)
density_df.round(1)  # 四舍五入到 1 位小数，表格更易读

,ward,population,area_km2,density
1,Higashi,87230,7.7,11313.9
0,Naka,99120,9.4,10567.2
13,Showa,108650,10.9,9931.4
5,Mizuho,106340,11.2,9477.7
2,Kita,165340,17.5,9431.8
14,Chikusa,164330,18.2,9039.1
3,Nishi,153210,17.9,8540.1
4,Nakamura,137800,16.3,8454.0
6,Atsuta,66780,8.2,8143.9
11,Meito,166850,21.5,7756.9


In [30]:
median_area = df["area_km2"].median()           # 所有区面积的中位数（返回一个标量）
big_wards = df[df["area_km2"] > median_area]   # 布尔索引：True/False 掩码，仅保留面积超过中位数的行
big_wards[["ward", "area_km2"]].sort_values("area_km2", ascending=False)  # 选两列并按面积降序展示

,ward,area_km2
7,Minato,45.64
10,Midori,37.91
9,Moriyama,34.01
15,Nakagawa,32.02
12,Tempaku,21.58
11,Meito,21.51
8,Minami,18.46
14,Chikusa,18.18


In [31]:
# 在真实数据上验证 add_density，并按密度降序展示前 5 行
(
    add_density(df)                              # 调用 src/ 里的函数，为 df 新增 density 列（原 df 不变）
    .sort_values("density", ascending=False)     # 按密度从高到低排序
    .head()                                      # 只看 Top 5，快速确认函数输出是否符合预期
)

,ward,population,area_km2,lat,lon,density
1,Higashi,87230,7.71,35.1819,136.9304,11313.878080
0,Naka,99120,9.38,35.1681,136.9066,10567.164179
13,Showa,108650,10.94,35.1494,136.9294,9931.444241
5,Mizuho,106340,11.22,35.1294,136.9376,9477.718360
2,Kita,165340,17.53,35.2086,136.9161,9431.831147
